# Fan Impact Pathway 온톨로지 — 전략 적용 노트북

목원님이 업로드하신 전략보고서(`docs/FANDOM_LDA_ONTOLOGY_STRATEGY.md` — 원본
「Fandom-omics LDA 고도화 전략」)가 제안하는 **K=Topic / F=Fan Impact Pathway /
FactorShare=Fan Factor Profile / Top-2 Factor 조합=Fan Persona / Loyalty·Spillover=
Impact Magnitude** 재정의를, 이번 세션에서 복구된 실제 데이터
(`data/v6_r22_snapshot/`, v7 라운드22 시점)에 적용해본다.

> **⚠️ 시점 불일치 주의**
> 전략문서 자체가 기준으로 삼은 수치는 코퍼스 5,794건·K=8·**M=5**·silhouette=0.202이고,
> 이 노트북이 실제로 쓰는 복구 스냅샷은 코퍼스 5,612건·K=8·**M=6**·silhouette=0.154다(라운드22
> 시점). 최종 제출 보고서는 또 다른 시점(코퍼스 10,020건·K=10·M=5·silhouette=0.267)이다.
> 세 시점이 서로 다르므로, 이 노트북의 실측 결과가 전략문서의 예시 표(6절 Persona 조합, 9절
> BTS·임영웅·리센느 비교표)와 다르게 나오는 지점을 각 셀에서 명시적으로 표시한다 — 이건
> 오류가 아니라 "서로 다른 코퍼스 스냅샷을 비교하고 있다"는 사실 자체를 보여주는 것이다.


## 0. 환경 설정 및 실데이터 로딩

In [1]:

import json
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

DATA_DIR = "../data/v6_r22_snapshot"

with open(f"{DATA_DIR}/fandoms_v3_100.json", encoding="utf-8") as f:
    fandoms_raw = json.load(f)
with open(f"{DATA_DIR}/lda_v6_diagnostics.json", encoding="utf-8") as f:
    diagnostics = json.load(f)
with open(f"{DATA_DIR}/fandom_scores_v6.json", encoding="utf-8") as f:
    scores = json.load(f)

scores_by_fandom = {d["fandom"]: d for d in scores}

print(f"팬덤 수: {len(fandoms_raw)}")
print(f"기록된 진단값: K={diagnostics['selected_k']}, "
      f"M={diagnostics['selected_m_meta_factors']}, "
      f"silhouette={diagnostics['meta_factor_silhouette']}")
print("(전략문서 기준값: K=8, M=5, silhouette=0.202 — M과 silhouette이 다름에 유의)")


팬덤 수: 100
기록된 진단값: K=8, M=6, silhouette=0.154
(전략문서 기준값: K=8, M=5, silhouette=0.202 — M과 silhouette이 다름에 유의)


## 1. 온톨로지 정의 (전략보고서 핵심 제안, 그대로 채택)

| 구분 | 현재 해석 | 고도화 해석 | 질문 |
|---|---|---|---|
| K | 세부 토픽/군집 | 행동·담론 Topic | 무엇을 하는가? |
| F | Meta Factor | Fan Impact Pathway | 어디로 전이되는가? |
| FactorShare | 요인별 근거 비중 | Fan Factor Profile | 어떤 경로가 강한가? |
| Top-2 Factor 조합 | (부재) | Fan Persona | 어떤 팬덤인가? |
| Loyalty/Spillover | 종합 점수 | Impact Magnitude(경로별 분해) | 얼마나 강한가? |


In [2]:

# 전략보고서 3.1절 — F Impact Pathway 권장 명칭(설계 제안, 5개 경로 기준)
F_IMPACT_PATHWAY = pd.DataFrame([
    {"F": "F1", "명칭": "팬덤결속 경로", "경로": "Fan → Fan",
     "대표신호": "팬클럽·기부·응원", "질문": "얼마나 강하게 결속되는가?"},
    {"F": "F2", "명칭": "직접소비 경로", "경로": "Fan → Market",
     "대표신호": "앨범·티켓·굿즈·판매", "질문": "충성도가 구매력으로 전환되는가?"},
    {"F": "F3", "명칭": "현장경제 경로", "경로": "Fan → Event → Local",
     "대표신호": "콘서트·투어·관객·숙박·교통", "질문": "팬의 이동이 지역경제로 이어지는가?"},
    {"F": "F4", "명칭": "산업전이 경로", "경로": "Fan → Brand/Industry",
     "대표신호": "광고·브랜드·앰버서더", "질문": "팬덤 영향력이 산업으로 전이되는가?"},
    {"F": "F5", "명칭": "대중·글로벌 확산 경로", "경로": "Fan → Media → Mass/Global",
     "대표신호": "차트·방송·유튜브·해외활동", "질문": "대중·해외시장으로 확장되는가?"},
])
F_IMPACT_PATHWAY


,F,명칭,경로,대표신호,질문
0,F1,팬덤결속 경로,Fan → Fan,팬클럽·기부·응원,얼마나 강하게 결속되는가?
1,F2,직접소비 경로,Fan → Market,앨범·티켓·굿즈·판매,충성도가 구매력으로 전환되는가?
2,F3,현장경제 경로,Fan → Event → Local,콘서트·투어·관객·숙박·교통,팬의 이동이 지역경제로 이어지는가?
3,F4,산업전이 경로,Fan → Brand/Industry,광고·브랜드·앰버서더,팬덤 영향력이 산업으로 전이되는가?
4,F5,대중·글로벌 확산 경로,Fan → Media → Mass/Global,차트·방송·유튜브·해외활동,대중·해외시장으로 확장되는가?


## 2. K→F 매핑 현황 점검 (실측, `lda_v6_diagnostics.json` 그대로 사용)

전략문서 4절·11절이 제안하는 "Topic-Factor 매핑률 + 미매핑 경보" QA를 실제 기록된 진단값에
적용한다. 이 셀은 refit 없이 **기록된 값 그대로**를 사용하므로 100% 실측이다.


In [3]:

topic_to_factor = diagnostics["topic_to_factor"]       # {topic_id: factor_id}
factor_labels = diagnostics["factor_labels"]           # {factor_id: label}
topics_top_words = diagnostics["topics_top_words"]     # {topic_id: [keywords]}
factor_top_words = diagnostics["factor_top_words"]     # {factor_id: [keywords]}

rows = []
for topic_id, factor_id in topic_to_factor.items():
    rows.append({
        "topic_id": f"K{int(topic_id)+1}",
        "topic_keywords": "·".join(topics_top_words[topic_id][:6]),
        "mapped_factor": f"F{int(factor_id)+1}",
        "factor_label(기록된 라벨)": factor_labels[str(factor_id)],
        "factor_keywords": "·".join(factor_top_words[str(factor_id)][:6]),
    })

mapping_df = pd.DataFrame(rows).sort_values("topic_id").reset_index(drop=True)

n_topics = len(topics_top_words)
n_mapped = len(topic_to_factor)
print(f"QA: 매핑률 = {n_mapped}/{n_topics} ({n_mapped/n_topics*100:.0f}%), "
      f"미매핑 Topic = {n_topics - n_mapped}개")
mapping_df


QA: 매핑률 = 8/8 (100%), 미매핑 Topic = 0개


,topic_id,topic_keywords,mapped_factor,factor_label(기록된 라벨),factor_keywords
0,K1,기록·1위·앨범·발매·차트·데뷔,F4,소비력형(초동·판매·앨범)·데뷔형,기록·1위·앨범·발매·차트·데뷔
1,K2,콘서트·공연·티켓·보도·투어·매진,F2,현장경제형(콘서트·투어·매진),콘서트·공연·보도·티켓·투어·매진
2,K3,예능·출연·드라마·지역·에서·프로그램,F6,미디어노출형(방송·조회수),예능·출연·드라마·지역·에서·프로그램
3,K4,공식·팬클럽·팬덤·데뷔·기부·2025년,F5,결속형(팬클럽·기부·커뮤니티),공식·팬클럽·팬덤·데뷔·기부·2025년
4,K5,tour·concert·sold·japan·world·album,F1,소비력형(초동·판매·앨범),tour·concert·sold·japan·world·album
5,K6,보도·콘서트·무대·공연·기사·일본,F2,현장경제형(콘서트·투어·매진),콘서트·공연·보도·티켓·투어·매진
6,K7,fan·music·official·club·brand·awards,F1,소비력형(초동·판매·앨범),tour·concert·sold·japan·world·album
7,K8,브랜드·앰버서더·발탁·광고·모델·수상,F3,브랜드·상업형(광고·앰버서더),브랜드·앰버서더·발탁·광고·모델·수상


### 2-1. 전략문서가 진단한 문제의 실제 사례

전략보고서 1.2절은 "K와 F의 역할이 겹쳐 보인다"고 진단한다. 위 표에서 실제로 이 현상이
보인다 — 예를 들어 K7(`fan·music·official·club·brand·awards`, 결속·브랜드가 섞인 키워드)이
K5(`tour·concert·sold·japan·world`, 월드투어/해외판매 키워드)와 같은 Factor로 묶이면서,
Factor의 대표 키워드에는 K7의 "club·brand·awards" 성격이 드러나지 않는다 — 이는 자동
키워드매칭 라벨링이 병합된 여러 Topic 중 하나의 어휘로 쏠릴 수 있음을 보여주는 실측 사례이며,
전략문서가 "라벨은 대표 Topic+키워드+근거문장을 함께 검토해 부여해야 한다"(3.2절)고 권고한
이유를 뒷받침한다.


## 3. Topic Card 포맷 데모 (일러스트레이션 — 신규 refit)

전략문서 2.2절이 제안하는 "Topic Card"(대표 키워드+대표 근거문장+대표 팬덤+연결 Factor)를
만들려면 문장별 topic 배정이 필요한데, 복구된 진단 파일에는 문장 단위 배정이 저장되어 있지
않다. 아래는 실제 근거문장 코퍼스(`fandoms_v3_100.json`, 5,612건)에 K=8 LDA를 **새로
refit**해서 Topic Card 포맷을 시연한 것이다.

> ⚠️ LDA는 라운드마다 토픽 순서·경계가 조금씩 달라질 수 있어, 아래 refit의 토픽 번호가 위
> 2번 셀의 기록된 진단값과 1:1로 정확히 대응한다고 보장할 수 없다. 포맷 자체를 보여주는
> 용도로만 사용할 것.


In [4]:

bullets = []
for entry in fandoms_raw:
    fandom = entry["fandom"]
    for bullet_type in ("loyalty", "spillover"):
        for b in entry.get(bullet_type, []):
            bullets.append({"fandom": fandom, "text": b.get("t", "")})

bullets_df = pd.DataFrame(bullets)
print(f"refit용 문장 수: {len(bullets_df)}")

vectorizer = CountVectorizer(token_pattern=r"(?u)\b\w{2,}\b", min_df=3, max_df=0.6)
dtm = vectorizer.fit_transform(bullets_df["text"])
vocab = np.array(vectorizer.get_feature_names_out())

K_REFIT = diagnostics["selected_k"]  # 기록된 K=8과 동일하게 맞춤(직접 비교 목적)
lda = LatentDirichletAllocation(n_components=K_REFIT, random_state=0, max_iter=30,
                                 learning_method="online", batch_size=512)
doc_topic = lda.fit_transform(dtm)
bullets_df["topic"] = doc_topic.argmax(axis=1)

def top_keywords(phi_row, n=8):
    idx = np.argsort(phi_row)[::-1][:n]
    return vocab[idx].tolist()

phi = lda.components_ / lda.components_.sum(axis=1, keepdims=True)
print("refit된 토픽별 상위 키워드:")
for k in range(K_REFIT):
    print(f"  topic_{k}: {'·'.join(top_keywords(phi[k]))}")


refit용 문장 수: 5612


refit된 토픽별 상위 키워드:
  topic_0: 콘서트·단독·공연·티켓·태국·보도·서울·투어
  topic_1: 위해·기부·기부했다·le·피해·sserafim·relief·지원
  topic_2: 인도네시아·1위를·보도했다·매체·보도·일본·차트·한국
  topic_3: 2026년·2025년·함께·지역·축제·부산·5월·무대에
  topic_4: the·in·and·of·on·for·to·with
  topic_5: 브랜드·앰버서더로·글로벌·모델로·광고·발탁·2025·앰버서더
  topic_6: 빌보드·유튜브·누적·만에·공식·스트리밍·기록했다·200
  topic_7: 데뷔·팬클럽·공식·앨범·발매·기록을·기준·2026년


In [5]:

def build_topic_card(topic_idx, n_examples=3, n_top_fandoms=5):
    sub = bullets_df[bullets_df["topic"] == topic_idx]
    kws = top_keywords(phi[topic_idx], n=12)
    examples = sub["text"].drop_duplicates().head(n_examples).tolist()
    top_fandoms = (sub.groupby("fandom").size().sort_values(ascending=False)
                   .head(n_top_fandoms))
    return {
        "topic_id": f"refit_K{topic_idx+1}",
        "n_bullets": len(sub),
        "대표_키워드": kws,
        "대표_근거문장(최대3)": examples,
        "대표_팬덤_Top5": top_fandoms.to_dict(),
    }

# 데모: refit 토픽 중 문장이 가장 많이 배정된 토픽 하나의 Topic Card를 출력
busiest_topic = bullets_df["topic"].value_counts().idxmax()
card = build_topic_card(busiest_topic)
import pprint
pprint.pprint(card)


{'n_bullets': 1122,
 'topic_id': 'refit_K5',
 '대표_근거문장(최대3)': ['High School Rapper 3와 Show Me the Money 11을 3년 내에 연달아 석권해 '
                  "'전무후무한' 이중 우승 기록을 세웠다",
                  'ชาว V.I.P เตรียมตัวให้พร้อม มาเจอ G-Dragon '
                  'แลนดิ้งสู่เมืองไทยในงาน"k-star spark in Bangkok 2025" '
                  'วันที่ 22 กุมภาพันธ์นี้',
                  'Theo số liệu ban tổ chức công bố, đêm nhạc hút gần 40.000 '
                  'fan.'],
 '대표_키워드': ['the',
            'in',
            'and',
            'of',
            'on',
            'for',
            'to',
            'with',
            'fan',
            'first',
            'tour',
            'at'],
 '대표_팬덤_Top5': {'BLACKPINK': 57,
                'BTS': 23,
                'LE SSERAFIM': 25,
                'NewJeans': 40,
                'TWICE': 28}}


## 4. Factor-specific Impact (전략문서 7절 산식, 100% 실측)

`fandom_scores_v6.json`에 이미 팬덤별 `factor_share`(F코드별 근거 비중)와
`loyalty_score`/`spillover_score`가 있으므로, refit 없이 그대로 전략문서 7절의 산식을
적용할 수 있다.

```
Factor-specific Loyalty   = FactorShare × Loyalty
Factor-specific Spillover = FactorShare × Spillover
```


In [6]:

def factor_specific_impact(fandom_name):
    d = scores_by_fandom[fandom_name]
    loyalty = d["loyalty_score"]
    spillover = d["spillover_score"]
    rows = []
    for factor_label, share in d["factor_share"].items():
        rows.append({
            "fandom": fandom_name,
            "factor_label": factor_label,
            "factor_share": round(share, 4),
            "factor_specific_loyalty": round(share * loyalty, 4),
            "factor_specific_spillover": round(share * spillover, 4),
        })
    return pd.DataFrame(rows).sort_values("factor_share", ascending=False)

all_rows = []
for fandom_name in scores_by_fandom:
    all_rows.append(factor_specific_impact(fandom_name))
factor_impact_all = pd.concat(all_rows, ignore_index=True)

# 검증: 팬덤별 factor_share 합 ≈ 1.0
share_sums = factor_impact_all.groupby("fandom")["factor_share"].sum()
bad = share_sums[(share_sums - 1.0).abs() > 0.02]
print(f"factor_share 합계가 1.0에서 0.02 넘게 벗어난 팬덤: {len(bad)}개")
print(f"전체 (팬덤 x factor) 행 수: {len(factor_impact_all)}")
factor_impact_all.head(10)


factor_share 합계가 1.0에서 0.02 넘게 벗어난 팬덤: 0개
전체 (팬덤 x factor) 행 수: 600


,fandom,factor_label,factor_share,factor_specific_loyalty,factor_specific_spillover
0,BTS,현장경제형(콘서트·투어·매진),0.2920,0.2476,0.2920
1,BTS,소비력형(초동·판매·앨범),0.2700,0.2290,0.2700
2,BTS,소비력형(초동·판매·앨범)·데뷔형,0.1614,0.1369,0.1614
3,BTS,미디어노출형(방송·조회수),0.1439,0.1220,0.1439
4,BTS,결속형(팬클럽·기부·커뮤니티),0.0855,0.0725,0.0855
5,BTS,브랜드·상업형(광고·앰버서더),0.0472,0.0400,0.0472
6,TWICE,소비력형(초동·판매·앨범),0.2964,0.2733,0.2543
7,TWICE,현장경제형(콘서트·투어·매진),0.2856,0.2633,0.2450
8,TWICE,소비력형(초동·판매·앨범)·데뷔형,0.1827,0.1684,0.1568
9,TWICE,브랜드·상업형(광고·앰버서더),0.1142,0.1053,0.0980


### 4-1. BTS·임영웅·리센느 실측 Factor-specific Impact

전략문서 7절/9절의 BTS·임영웅·리센느 비교표는 "표현 방식의 예시"라고 명시되어 있다(실제
재산출 결과가 아님). 아래는 이번에 복구된 실측 `fandom_scores_v6.json` 기준으로 세 팬덤의
대표 Factor와 Factor-specific Impact를 실제로 계산한 결과다 — 전략문서 예시표와 다를 수
있으며, 다르다면 그것이 "이 스냅샷 시점의 실제 수치"다.


In [7]:

HIGHLIGHT = ["BTS", "임영웅", "리센느(RESCENE)"]
for name in HIGHLIGHT:
    if name not in scores_by_fandom:
        print(f"[스킵] {name}: 이 스냅샷에 없음")
        continue
    d = scores_by_fandom[name]
    top_factor = d["dominant_factor"]
    print(f"\n=== {name} ===")
    print(f"loyalty_score={d['loyalty_score']}, spillover_score={d['spillover_score']}, "
          f"dominant_factor={top_factor}")
    fi = factor_specific_impact(name)
    display_cols = ["factor_label", "factor_share", "factor_specific_loyalty", "factor_specific_spillover"]
    print(fi[display_cols].head(3).to_string(index=False))



=== BTS ===
loyalty_score=0.848, spillover_score=1.0, dominant_factor=현장경제형(콘서트·투어·매진)
      factor_label  factor_share  factor_specific_loyalty  factor_specific_spillover
  현장경제형(콘서트·투어·매진)        0.2920                   0.2476                     0.2920
    소비력형(초동·판매·앨범)        0.2700                   0.2290                     0.2700
소비력형(초동·판매·앨범)·데뷔형        0.1614                   0.1369                     0.1614

=== 임영웅 ===
loyalty_score=0.866, spillover_score=0.576, dominant_factor=현장경제형(콘서트·투어·매진)
      factor_label  factor_share  factor_specific_loyalty  factor_specific_spillover
  현장경제형(콘서트·투어·매진)        0.2794                   0.2420                     0.1609
소비력형(초동·판매·앨범)·데뷔형        0.1993                   0.1726                     0.1148
  결속형(팬클럽·기부·커뮤니티)        0.1906                   0.1651                     0.1098

=== 리센느(RESCENE) ===
loyalty_score=0.465, spillover_score=0.472, dominant_factor=현장경제형(콘서트·투어·매진)
      factor_label  factor_share  factor_sp

## 5. Fan Persona 데모 (Top-2 Factor 조합) — M=6 기준으로 조정

전략문서 6절의 Persona 조합표는 F가 5개(F1~F5)라는 가정 위에 만들어진 설계 예시다. 이번
복구 스냅샷은 실제로는 **M=6**(factor_labels 6종)이므로, 문서의 5개 고정 조합표를 그대로
적용할 수 없다 — 대신 같은 아이디어(Top-2 Factor 조합 → Persona 후보)를 이 스냅샷의 실제
6개 라벨 기준으로 재현해, 어떤 조합이 실제로 나타나는지 집계한다.


In [8]:

def top2_factors(fandom_name):
    d = scores_by_fandom[fandom_name]
    share = d["factor_share"]
    top2 = sorted(share.items(), key=lambda kv: -kv[1])[:2]
    return tuple(sorted(label for label, _ in top2))

combo_counts = {}
for fandom_name in scores_by_fandom:
    combo = top2_factors(fandom_name)
    combo_counts[combo] = combo_counts.get(combo, 0) + 1

combo_df = pd.DataFrame(
    [{"top2_factor_combo": " + ".join(c), "n_fandoms": n} for c, n in combo_counts.items()]
).sort_values("n_fandoms", ascending=False).reset_index(drop=True)

print(f"이론상 가능한 조합 수(M=6, 순서무관 2개조합) = {6*5//2}개, "
      f"실제 실현된 조합 수 = {len(combo_df)}개")
combo_df


이론상 가능한 조합 수(M=6, 순서무관 2개조합) = 15개, 실제 실현된 조합 수 = 6개


,top2_factor_combo,n_fandoms
0,소비력형(초동·판매·앨범) + 현장경제형(콘서트·투어·매진),56
1,소비력형(초동·판매·앨범)·데뷔형 + 현장경제형(콘서트·투어·매진),27
2,결속형(팬클럽·기부·커뮤니티) + 현장경제형(콘서트·투어·매진),11
3,브랜드·상업형(광고·앰버서더) + 현장경제형(콘서트·투어·매진),4
4,결속형(팬클럽·기부·커뮤니티) + 소비력형(초동·판매·앨범),1
5,소비력형(초동·판매·앨범) + 소비력형(초동·판매·앨범)·데뷔형,1


## 6. QA 요약 (전략문서 11절 체크리스트 적용 결과)

| 체크 항목 | 결과 |
|---|---|
| Topic→Factor 매핑률 | 위 2번 셀 참고 (기록된 진단 기준) |
| factor_share 합계 검증 | 위 4번 셀 참고 |
| K/M/silhouette 시점 일치 여부 | 불일치 — 아래 최종 노트 참고 |


## 최종 노트 — 전략문서 예시와 실측값이 다른 지점

1. **기준 수치 자체가 다른 시점이다.** 전략문서: 코퍼스 5,794건·K=8·M=5·silhouette=0.202.
   이 노트북: 코퍼스 5,612건·K=8·**M=6**·silhouette=0.154 (v7 라운드22). 최종 제출본:
   코퍼스 10,020건·K=10·M=5·silhouette=0.267. 세 값 모두 다른 시점의 정직한 실측값이며,
   어느 하나가 "틀린" 것이 아니라 코퍼스가 계속 자란 서로 다른 단계다.
2. **6절 Persona 조합표는 M=5 가정이라 이 스냅샷(M=6)에 그대로 적용 불가** — 5번 셀에서
   같은 아이디어를 M=6 기준으로 재현했다.
3. **9절 BTS·임영웅·리센느 비교표는 문서 스스로 "표현 방식의 예시"라고 명시** — 4-1번 셀의
   실측값으로 대체해서 봐야 한다.
4. **최종 제출 보고서(`docs/METHODOLOGY.md`)의 실현 Persona 4종**(집단동원형·원정소비형·
   현장상업형·글로벌투어형, F1~F5 5개 경로 기준)은 이 노트북이 쓰는 v6 라운드22 스냅샷이
   아니라 최종 v7-40 동결 스냅샷에서 나온 결과이므로, 이 노트북의 5번 셀 결과와 직접 비교하지
   말 것.
